In [1]:
import celltypist
from datetime import date
import hisepy
import numpy as np
import os
import pandas as pd
import scanpy as sc

### Retrieve 10x Genomics L1000 Probe gene list

In [2]:
immune_uuid = 'eb70eddd-93d8-4d75-a087-9a98d74db0b1'
immune_file = hisepy.cache_files([immune_uuid])[0]

In [3]:
immune_probes = pd.read_csv(immune_file)

In [4]:
immune_genes = immune_probes['gene'].unique().tolist()

In [5]:
l1000_uuid = 'b9e655f5-05f7-4138-b3c2-2097bba88d61'
l1000_file = hisepy.cache_files([l1000_uuid])[0]

In [6]:
l1000_probes = pd.read_csv(l1000_file)

In [7]:
l1000_genes = l1000_probes['gene'].unique().tolist()

In [8]:
flex_genes = l1000_genes + immune_genes

In [9]:
flex_genes = list(set(l1000_genes).union(set(immune_genes)))

In [10]:
len(flex_genes)

1968

### Helper functions

In [11]:
def read_adata_uuid(h5ad_uuid):
    h5ad_file = hisepy.cache_files([h5ad_uuid])[0]
    adata = sc.read_h5ad(h5ad_file)
    return adata

In [12]:
def resample_anndata_min_max(adata, label_column, max_cells=None, min_cells=None, random_state = 3030):
    """
    Resamples an AnnData object based on the cell labels, with the option to resample with 
    replacement for classes below a specified threshold.

    Parameters:
    ad (AnnData): The AnnData object to be resampled.
    label_column (str): The column in ad.obs where the labels are stored.
    max_cells (int, optional): The maximum number of cells to keep per label. If None, no upper limit is applied.
    min_cells (int, optional): The minimum number of cells below which resampling with replacement occurs. If None, no lower limit is applied.
    random_state (int, default = 3030): An integer used to set the state of the numpy.random.Generator
    
    Returns:
    AnnData: The resampled AnnData object.
    """
    
    labels = adata.obs[label_column].unique()

    subsets = []

    rng = np.random.default_rng(random_state)
        
    for label in labels:
        # Subset AnnData object for the current label
        subset = adata.obs[adata.obs[label_column] == label]
    
        # Resample with replacement if the number of cells is below min_cells and min_cells is defined
        if min_cells is not None and subset.shape[0] < min_cells:
            subset = subset.sample(min_cells, replace = True, random_state = rng)
        # Resample without replacement if the number of cells is greater than max_cells and max_cells is defined
        elif max_cells is not None and subset.shape[0] > max_cells:
            subset = subset.sample(max_cells, replace = False, random_state = rng)
    
        subsets.append(subset)

    # Concatenate all subsets
    resampled_obs = pd.concat(subsets)
    
    resampled_adata = adata[resampled_obs.index]
    resampled_adata.obs_names_make_unique()

    return resampled_adata

In [13]:
label_column = 'AIFI_L2'
max_cell_number = 100000

## Read clean, annotated dataset

In [14]:
h5ad_uuid = '4a4962b9-6e11-4f7a-ad0c-f32e49f39e34'

In [15]:
adata = read_adata_uuid(h5ad_uuid)

In [16]:
adata.shape

(1821725, 33538)

In [17]:
adata.obs[label_column].value_counts()

AIFI_L2
Naive CD4 T cell         378071
Memory CD4 T cell        321788
CD14 monocyte            269328
Memory CD8 T cell        183096
CD56dim NK cell          133881
Naive CD8 T cell         121167
Naive B cell              86711
gdT                       50587
Memory B cell             47886
MAIT                      46143
CD16 monocyte             45920
Treg                      39087
cDC2                      14235
Intermediate monocyte     12671
Transitional B cell       12555
Effector B cell           11329
CD56bright NK cell        11055
Platelet                   7903
pDC                        7587
CD8aa                      5737
Proliferating NK cell      2825
DN T cell                  2349
Proliferating T cell       2320
Plasma cell                2151
Progenitor cell            1526
Erythrocyte                1508
cDC1                        943
ILC                         844
ASDC                        522
Name: count, dtype: int64

In [18]:
adata = adata.raw.to_adata()
adata.shape

(1821725, 33538)

### Limit to genes available in 10x L1000 Flex Probes

In [19]:
keep_var = adata.var.index.isin(flex_genes)
sum(keep_var)

np.int64(1932)

In [20]:
adata = adata[:,keep_var]
adata.shape

(1821725, 1932)

In [21]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

## Generate model using selected features

In [22]:
model_fs = celltypist.train(
    adata, 
    label_column, 
    n_jobs = 42,
    max_iter = 100,
    check_expression = False
)

🍳 Preparing data before training
✂️ 20 non-expressed genes are filtered out
🔬 Input data has 1821725 cells and 1912 genes
⚖️ Scaling input data
🏋️ Training data using logistic regression
✅ Model training done!


## Write outputs for storage

In [23]:
out_dir = 'output'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

In [24]:
out_model = 'output/ref_pbmc_clean_celltypist_model_l1000-immune-features_{l}_{d}.pkl'.format(
    l = label_column,
    d = date.today()
)

model_fs.write(out_model)

## Upload model to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [25]:
study_space_uuid = '64097865-486d-43b3-8f94-74994e0a72e0'
title = 'PBMC Reference {l} CellTypist Model L1000 and Immune Features {d}'.format(
    l = label_column,
    d = date.today()
)

In [26]:
in_files = [immune_uuid, l1000_uuid, h5ad_uuid]

In [27]:
in_files

['eb70eddd-93d8-4d75-a087-9a98d74db0b1',
 'b9e655f5-05f7-4138-b3c2-2097bba88d61',
 '4a4962b9-6e11-4f7a-ad0c-f32e49f39e34']

In [28]:
out_files = [out_model]

In [29]:
out_files

['output/ref_pbmc_clean_celltypist_model_l1000-immune-features_AIFI_L2_2025-03-13.pkl']

In [30]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files
)

checking if conda environment can compile...


{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': '3710b90d-fd6b-48f1-a660-0f5091c815ed',
 'ProcessId': '202b170e-7cb6-442c-a3cf-573aac2305c1',
 'WorkflowId': 'bdeb51ab-7e44-45e3-93ee-e5cff6d00eda',
 'FileIds': ['9cc3e9df-4003-4471-872c-7c86cc5a7aac']}

In [31]:
import session_info
session_info.show()